# 🚁 Notebook 04 — Building the P Controller

### From clumsy on/off to a smooth, proportional push

In Notebook 03 the drone held its altitude with **bang-bang** control — but it buzzed forever,
because it only knew two moves: *full blast* or *nothing*. This notebook fixes that with the first
letter of PID: **P**, for **Proportional**. The idea is almost obvious once you hear it:

> **Push in proportion to how wrong you are.** Far below target → push hard. Almost there → push
> gently. Right on target → don't push at all.

That single rule gives smooth, sensible behaviour — and also reveals a subtle problem (a permanent
little error) that sets up Notebook 05.

---

## 🎯 Learning objectives
- Build a **proportional controller** by hand: `thrust = Kp * error`.
- Tune the gain **Kp** and feel the difference between **small** (sluggish) and **large** (overshoot, oscillation).
- Define and *measure* **rise time, overshoot, oscillation, settling time**.
- Discover the **steady-state error** that P alone can never remove.

## 🧩 Prerequisites
Notebooks 01–03 (Euler simulation, forces/gravity/thrust, feedback, error, bang-bang).

## ⏱️ Estimated time
About **45–60 minutes**.

## 🧠 Concepts you know so far
Euler simulation · force/mass/gravity/thrust · net force · open vs closed loop · sensor · target · **error** · bang-bang feedback

## 🔜 Concepts you'll learn here
Proportional gain (Kp) · proportional control law · rise time · overshoot · oscillation · settling time · steady-state error


### 🔁 Quick recap

Feedback loop, every time step: **measure → error = target − current → decide thrust → drone moves.**
Bang-bang decided thrust with a harsh `if`: max below target, zero above. It worked but wobbled.
We'll replace that harsh `if` with one smooth line. Run setup first.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80
plt.rcParams["figure.figsize"] = (8, 4)

from ipywidgets import interact, FloatSlider, IntSlider, Checkbox

try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready!")


## 1. The proportional idea (intuition first)

Imagine filling a glass of water 🥛. When it's nearly empty you open the tap wide. As it fills, you
*ease off*. When it's full, you shut it. You naturally pour **in proportion to how far from full**
it is. That's proportional control.

For our drone, "how far from target" **is the error**. So:

> **thrust = Kp × error**

- Big error (far below target) → big thrust → strong climb.
- Small error (almost there) → small thrust → gentle approach.
- Zero error → zero thrust from the P term.

`Kp` is the **proportional gain** — a dial that sets *how aggressively* we react to each metre of
error. That's the whole controller. One line. Let's build it.


## 2. Build it, one line at a time

Here is our drone loop from before, with the harsh bang-bang `if` swapped for the single smooth
proportional line. Read the comments — the only new idea is the `thrust = Kp * error` line.


In [ ]:
def simulate_P(Kp, target=10.0, mass=1.0, g=9.8, start_height=0.0,
               dt=0.02, total_time=12.0, max_thrust=30.0):
    """Drone controlled by a PURE proportional controller: thrust = Kp * error."""
    x, v, t = start_height, 0.0, 0.0
    times, xs, thrusts = [], [], []
    for _ in range(int(total_time / dt)):
        # ---- SENSE + COMPARE ----
        error = target - x                      # how far below target are we?
        # ---- DECIDE (the proportional controller: ONE line) ----
        thrust = Kp * error                     # push in proportion to the error
        thrust = max(0.0, min(thrust, max_thrust))   # motors can't pull down or exceed max
        # ---- ACT (physics from Notebook 02) ----
        net_force = thrust - mass * g           # thrust up, gravity down
        a = net_force / mass
        times.append(t); xs.append(x); thrusts.append(thrust)
        v = v + a * dt                          # Euler integrate
        x = x + v * dt
        if x < 0: x, v = 0.0, 0.0
        t = t + dt
    return np.array(times), np.array(xs), np.array(thrusts)

t, x, thr = simulate_P(Kp=3.0)
plt.figure(figsize=(8, 4.5))
plt.plot(t, x, color="royalblue", lw=2, label="drone altitude")
plt.axhline(10, color="seagreen", ls="--", label="target 10 m")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("A proportional controller (Kp = 3): smooth rise toward target")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
print("Notice how smoothly it climbs and eases in -- no bang-bang buzzing!")
print("But look closely: does it actually REACH the green line? (Hold that thought.)")


## 3. Tuning Kp: sluggish → crisp → overshoot → oscillation

`Kp` is the aggressiveness dial. Let's watch what happens as we turn it up. This is one of the most
important pictures in the whole course — study it.


In [ ]:
plt.figure(figsize=(9, 5))
for Kp, colour in [(0.5, "tab:blue"), (2.0, "tab:green"),
                   (6.0, "tab:orange"), (15.0, "tab:red")]:
    t, x, _ = simulate_P(Kp=Kp)
    plt.plot(t, x, color=colour, lw=2, label=f"Kp = {Kp}")
plt.axhline(10, color="black", ls="--", label="target")
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Effect of Kp:  low = slow & lazy,  high = fast but overshoots & oscillates")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

print("Kp = 0.5  -> very SLOW, lazy climb, stops well short of target.")
print("Kp = 2.0  -> reasonable, still stops a bit short.")
print("Kp = 6.0  -> fast, but now it OVERSHOOTS past the target and comes back.")
print("Kp = 15.0 -> too aggressive: big overshoot and OSCILLATION (bouncing up and down).")


👀 **The core trade-off of proportional control:** turning `Kp` up makes the drone respond faster,
but past a point it **overshoots** (flies past the target) and **oscillates** (bounces around it).
Too low and it's hopelessly slow. Good tuning lives in the middle — and we'll get precise language
for "good" next.


## 4. The vocabulary every control engineer uses

When we command a sudden jump to a target (a **step**) and watch the response, four numbers
describe how good it is. Learn these four words — you'll use them for the rest of your life:

- **Rise time** ⏱️ — how long to *first* get close to the target (we'll use "reach 90%").
- **Overshoot** 📈 — how far it flies *past* the target, as a % of the target.
- **Settling time** 🎯 — when it finally stays within a small band (±5%) of the target and *stops*
  wandering.
- **Steady-state error** ➖ — the leftover gap between where it settles and the target.

Let's label them on a real response.


In [ ]:
t, x, _ = simulate_P(Kp=8.0, total_time=10.0)
target = 10.0
peak = x.max(); peak_t = t[x.argmax()]
final = x[int(0.9*len(x)):].mean()

plt.figure(figsize=(9, 5))
plt.plot(t, x, color="royalblue", lw=2, label="altitude")
plt.axhline(target, color="seagreen", ls="--", label="target")
plt.axhspan(target*0.95, target*1.05, color="seagreen", alpha=0.12, label="+/-5% settle band")
# overshoot marker
plt.annotate("overshoot", xy=(peak_t, peak), xytext=(peak_t+1, peak+0.5),
             arrowprops=dict(arrowstyle="->")); plt.plot(peak_t, peak, "ro")
# steady-state error marker
plt.annotate(f"steady-state error = {target-final:.2f} m",
             xy=(t[-1], final), xytext=(t[-1]-4.5, final-2.2),
             arrowprops=dict(arrowstyle="->"))
plt.xlabel("time (s)"); plt.ylabel("altitude (m)")
plt.title("Reading a step response (Kp = 8)")
plt.legend(loc="lower right"); plt.grid(True, alpha=0.3); plt.show()

print(f"peak altitude   = {peak:.2f} m  (overshoot = {(peak-target)/target*100:.1f}%)")
print(f"settles near    = {final:.2f} m")
print(f"steady-state err= {target-final:.2f} m  <-- it never quite reaches 10 m!")


## 5. 🎬 Watch a P-controlled drone rise

See the smooth, eased approach (very different from bang-bang's buzzing). With this moderate `Kp`
it glides up and settles — but a hair below the line.


In [ ]:
t, x, thr = simulate_P(Kp=4.0, total_time=10.0)
frame_ids = np.linspace(0, len(t)-1, 120).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1); ax.set_ylim(0, 13); ax.set_xticks([])
ax.set_ylabel("altitude (m)"); ax.set_title("Proportional control (Kp = 4)")
ax.axhline(0, color="saddlebrown", lw=3)
ax.axhline(10, color="seagreen", ls="--", lw=2)
drone, = ax.plot([], [], "o", color="royalblue", markersize=26)
label = ax.text(-0.9, 12, "", fontsize=11)

def init(): drone.set_data([], []); label.set_text(""); return drone, label
def update(f):
    i = frame_ids[f]; drone.set_data([0], [x[i]])
    label.set_text(f"t={t[i]:.1f}s\nx={x[i]:.2f} m\nthrust={thr[i]:.1f} N")
    return drone, label

ani = animation.FuncAnimation(fig, update, frames=len(frame_ids), init_func=init,
                              blit=True, interval=45)
plt.close(fig); HTML(ani.to_jshtml())


## 6. Why P *always* stops a little short (the steady-state error)

This isn't a bug — it's baked into the math, and understanding *why* makes integral control
(Notebook 05) feel inevitable.

The drone can only hover when thrust exactly balances gravity: **thrust = m·g** (≈ 9.8 N). Our P
controller produces `thrust = Kp × error`. So to *hold* a hover, we need:

$$ K_p \times \text{error} = m\,g \quad\Longrightarrow\quad \text{error} = \frac{m\,g}{K_p} $$

Read that carefully: to generate enough thrust to fight gravity, **the error must be non-zero.** If
the error were zero, the thrust would be zero, and the drone would fall! So P *needs* a permanent
little error just to stay up. That leftover gap is the **steady-state error**, and it equals
`m·g / Kp`.

- Bigger `Kp` → smaller steady-state error (but more overshoot/oscillation).
- You can never fully remove it with P alone — only shrink it, at the cost of stability.

Let's verify the formula against the simulation.


In [ ]:
mass, g = 1.0, 9.8
print(f"{'Kp':>5} | {'predicted error m*g/Kp':>22} | {'measured error':>15}")
print("-"*50)
for Kp in [1, 2, 4, 8, 16]:
    t, x, _ = simulate_P(Kp=Kp, total_time=20.0)
    measured_error = 10.0 - x[int(0.9*len(x)):].mean()
    predicted = mass*g/Kp
    print(f"{Kp:>5} | {predicted:>22.3f} | {measured_error:>15.3f}")
print("\nThe formula error = m*g/Kp matches the simulation. Doubling Kp roughly halves the error,")
print("but you can never reach exactly zero with P alone. Integral control (next!) fixes this.")


## 7. 🎛️ Play with Kp yourself

Drag `Kp` and watch the whole trade-off live: too low = slow and far short; too high = overshoot
and oscillation. Try to find a `Kp` you'd call "nicely tuned," then notice it *still* settles below
the line. That stubborn gap is exactly what Notebook 05 removes.


In [ ]:
def explore_P(Kp=4.0, target=10.0):
    t, x, thr = simulate_P(Kp=Kp, target=target, total_time=12.0)
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
    a1.plot(t, x, color="royalblue", lw=2); a1.axhline(target, color="seagreen", ls="--")
    a1.set_ylabel("altitude (m)"); a1.set_ylim(0, target*1.6)
    a1.set_title(f"Kp = {Kp}   |   steady-state error approx {target - x[int(0.9*len(x)):].mean():.2f} m")
    a1.grid(True, alpha=0.3)
    a2.plot(t, thr, color="darkorange"); a2.set_ylabel("thrust (N)"); a2.set_xlabel("time (s)")
    a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

interact(explore_P,
         Kp=FloatSlider(min=0.3, max=20.0, step=0.3, value=4.0),
         target=FloatSlider(min=4.0, max=14.0, step=1.0, value=10.0));


## ✅ Summary
- **Proportional control:** `thrust = Kp × error`. Push in proportion to how wrong you are.
- **Kp** is the aggressiveness dial: too low → slow & far short; too high → overshoot & oscillation.
- Step-response vocabulary: **rise time, overshoot, settling time, steady-state error.**
- P alone leaves a **steady-state error of `m·g / Kp`**, because it needs a non-zero error to
  produce the thrust that fights gravity. Raising Kp shrinks it but never removes it.


## ⚠️ Common mistakes
- **Thinking a big enough Kp reaches the target exactly.** It only shrinks the error toward zero,
  never to zero — and huge Kp causes violent oscillation.
- **Forgetting the thrust clamp.** Real motors can't produce negative thrust or unlimited thrust.
- **Confusing overshoot with instability.** A little overshoot can be fine; growing oscillations
  are the real danger.

## 🧭 Mental model
> *"P is a spring: the further you are from target, the harder it pulls you back — but a spring
> alone settles wherever its pull balances gravity, which is slightly off-target."*

## 🌍 Real-world applications
Cruise control easing off as you near the set speed, a thermostat with proportional valves, camera
autofocus, and the innermost loop of essentially every real flight controller.


## 🧪 Exercises

**L1 — Observe.** In the comparison plot, which `Kp` gives the *smallest* steady-state error, and
what's the price you pay for it?
<details><summary>▸ Solution</summary>
The largest `Kp` (15) has the smallest steady-state gap, but pays with big overshoot and
oscillation. Smaller error, worse stability — the fundamental P trade-off.
</details>

**L2 — Modify.** In `simulate_P`, add a constant `wind = -2.0` N to `net_force`. Does the
steady-state error get bigger or smaller? Why?
<details><summary>▸ Hint</summary>
Now the controller must supply thrust to fight gravity <b>and</b> the downward wind.
</details>
<details><summary>▸ Solution</summary>
Bigger. To hover it now needs <code>thrust = m*g + 2</code>, so the required error grows to
<code>(m*g+2)/Kp</code>. Any extra steady load increases P's steady-state error — more motivation
for integral control.
</details>

**L3 — Predict.** With `Kp = 5`, `m = 1`, `g = 9.8`, what steady-state error do you expect? Predict,
then run `simulate_P(Kp=5, total_time=20)` and check.
<details><summary>▸ Solution</summary>
<code>m*g/Kp = 9.8/5 = 1.96 m</code>. The drone settles at about <code>10 - 1.96 = 8.04 m</code>.
</details>


## ❓ Quick quiz
**Q1.** Doubling Kp does what to the steady-state error?
<details><summary>▸ Answer</summary>Roughly **halves** it (error = m·g/Kp), but increases overshoot.</details>

**Q2.** Why can't a pure P controller sit exactly on target while hovering?
<details><summary>▸ Answer</summary>Because it needs a **non-zero error** to generate the thrust (m·g) that balances gravity. Zero error → zero thrust → it falls.</details>

**Q3.** Overshoot means…
<details><summary>▸ Answer</summary>The drone **flies past** the target before coming back.</details>

**Q4.** Very large Kp tends to cause…
<details><summary>▸ Answer</summary>**Oscillation** (and large overshoot) — the response becomes twitchy and unstable.</details>


## 🔭 Preview of Notebook 05 — *Integral Control*

We just proved P leaves a permanent little error. The fix: keep a **running total of the error over
time** and use it to add just enough extra thrust to close the gap — the **Integral (I)** term. It
patiently notices "we've been a bit short for a while" and ramps up until the error hits **exactly
zero**. You'll also meet its dark side — **windup** — and the standard cure, **anti-windup**. 🚁
